# Forest Cover Type Prediction

 predict the forest cover type (the predominant kind of tree cover) from strictly cartographic variables (as opposed to remotely sensed data). The actual forest cover type for a given 30 x 30 meter cell was determined from US Forest Service (USFS) Region 2 Resource Information System data. Independent variables were then derived from data obtained from the US Geological Survey and USFS. The data is in raw form (not scaled) and contains binary
columns of data for qualitative independent variables such as wilderness areas and soil type. 


#### Data Fields :

Data Fields
Elevation - Elevation in meters

Aspect - Aspect in degrees azimuth

Slope - Slope in degrees

Horizontal_Distance_To_Hydrology - Horz Dist to nearest surface water features

Vertical_Distance_To_Hydrology - Vert Dist to nearest surface water features

Horizontal_Distance_To_Roadways - Horz Dist to nearest roadway

Hillshade_9am (0 to 255 index) - Hillshade index at 9am, summer solstice

Hillshade_Noon (0 to 255 index) - Hillshade index at noon, summer solstice

Hillshade_3pm (0 to 255 index) - Hillshade index at 3pm, summer solstice

Horizontal_Distance_To_Fire_Points - Horz Dist to nearest wildfire ignition points

Wilderness_Area (4 binary columns, 0 = absence or 1 = presence) - Wilderness area designation

Soil_Type (40 binary columns, 0 = absence or 1 = presence) - Soil Type designation

Cover_Type (7 types, integers 1 to 7) - Forest Cover Type designation
The wilderness areas are:

1 - Rawah Wilderness Area
2 - Neota Wilderness Area
3 - Comanche Peak Wilderness Area
4 - Cache la Poudre Wilderness Area

The soil types are:

1 Cathedral family - Rock outcrop complex, extremely stony.

2 Vanet - Ratake families complex, very stony.

3 Haploborolis - Rock outcrop complex, rubbly.

4 Ratake family - Rock outcrop complex, rubbly.

5 Vanet family - Rock outcrop complex complex, rubbly.

6 Vanet - Wetmore families - Rock outcrop complex, stony.

7 Gothic family.

8 Supervisor - Limber families complex.

9 Troutville family, very stony.

10 Bullwark - Catamount families - Rock outcrop complex, rubbly.

11 Bullwark - Catamount families - Rock land complex, rubbly.

12 Legault family - Rock land complex, stony.

13 Catamount family - Rock land - Bullwark family complex, rubbly.

14 Pachic Argiborolis - Aquolis complex.

15 unspecified in the USFS Soil and ELU Survey.

16 Cryaquolis - Cryoborolis complex.

17 Gateview family - Cryaquolis complex.

18 Rogert family, very stony.

19 Typic Cryaquolis - Borohemists complex.

20 Typic Cryaquepts - Typic Cryaquolls complex.

21 Typic Cryaquolls - Leighcan family, till substratum complex.

22 Leighcan family, till substratum, extremely bouldery.

23 Leighcan family, till substratum - Typic Cryaquolls complex.

24 Leighcan family, extremely stony.

25 Leighcan family, warm, extremely stony.

26 Granile - Catamount families complex, very stony.

27 Leighcan family, warm - Rock outcrop complex, extremely stony.

28 Leighcan family - Rock outcrop complex, extremely stony.

29 Como - Legault families complex, extremely stony.

30 Como family - Rock land - Legault family complex, extremely stony.

31 Leighcan - Catamount families complex, extremely stony.

32 Catamount family - Rock outcrop - Leighcan family complex, extremely stony.

33 Leighcan - Catamount families - Rock outcrop complex, extremely stony.

34 Cryorthents - Rock land complex, extremely stony.

35 Cryumbrepts - Rock outcrop - Cryaquepts complex.

36 Bross family - Rock land - Cryumbrepts complex, extremely stony.

37 Rock outcrop - Cryumbrepts - Cryorthents complex, extremely stony.

38 Leighcan - Moran families - Cryaquolls complex, extremely stony.

39 Moran family - Cryorthents - Leighcan family complex, extremely stony.

40 Moran family - Cryorthents - Rock land complex, extremely stony

### import paskages

In [ ]:
import pandas as pd
import numpy as np
import xgboost 
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix,classification_report
from sklearn.model_selection import train_test_split , cross_val_score,KFold,GridSearchCV
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_selection import RFE

In [ ]:
#load data
df=pd.read_csv("/kaggle/input/forest-cover-type-prediction/train.csv" , index_col=["Id"])

In [ ]:
#load test data
df_test=pd.read_csv("/kaggle/input/forest-cover-type-prediction/test.csv" )

In [ ]:
# head data
df.head()

In [ ]:
#information data

print(df.info())

In [ ]:
# describtion data
print(df.describe())

In [ ]:
#name columns
df.columns

In [ ]:
#check is null data

df.isnull().sum()

#no missing data

In [ ]:
# shape data
df.shape

 The data contains 15,120 samples

In [ ]:
#describe  Cover_Type target

df['Cover_Type'].describe()

In [ ]:
# counts value  target 'Cover_Type'
class_label =df['Cover_Type'].value_counts()
class_label

 All classes are equal

To compare Average  features with target classes


In [ ]:
#To compare features with target classes
def comp_target(feature):
    """
    function compare mean features to lasse target [1-7]
    
    input : 
            name columns
    
    output: 
            dictionary enclude number class and mean 
    """
    
    comp={}
    for i in range(1,8):
        c= df[df["Cover_Type"] == i][feature].mean()
        comp[i]=c
    return comp

In [ ]:
#plot compare Average  features  with target

def plot_comp(fea_c_target ):
    """
    #plot compare Average  features  with target
    
    input:
            dictionary
    
    output:
            plot bar
    
    """
    
    
    plt.bar(fea_c_target.keys()  ,fea_c_target.values())
    plt.xlabel("Number Target ")
    plt.ylabel("Average  ")
    plt.show()


In [ ]:
# Elevation compare with target

Elevation_c_target = comp_target("Elevation")
Elevation_c_target

In [ ]:
#plot Average Elevation compare with target


plot_comp(Elevation_c_target)

 Elevation in meters to class 7 is biggest

In [ ]:
# Aspect compare with target

Aspect_c_target = comp_target("Aspect")
Aspect_c_target

In [ ]:
plot_comp(Aspect_c_target)

 Aspect in degrees azimuth  to class 6 is biggest

In [ ]:
# Horizontal_Distance_To_Hydrology compare with target

HDH_c_target = comp_target("Horizontal_Distance_To_Hydrology")
HDH_c_target

In [ ]:
plot_comp(HDH_c_target)

Horz Dist to nearest surface water features to class 7 is biggest

In [ ]:
# Slope compare with target

Slope_c_target = comp_target("Slope")
Slope_c_target

In [ ]:
plot_comp(Slope_c_target)

 Slope in degrees to class 3 is biggest

In [ ]:
# correlation  target  and other features in absolute 
correlations = df.corr()['Cover_Type'].drop('Cover_Type')
sorted_correlations = correlations.abs().sort_values(ascending=False)
sorted_correlations

It seems blurry so we will reduce the unimportant features

In [ ]:
#  bar to compare correlation between a target and  features with absolute 
# to be organized and easy to compare

plt.figure(figsize=(40, 15))
sns.barplot(x=sorted_correlations.index, y=sorted_correlations)
plt.xticks(rotation=90)
plt.xlabel('Features')
plt.ylabel('Absolute Correlation')
plt.title("Correlation data with target")
plt.show()

## Extract 5 additional features

This feature would calculate the Euclidean distance from each cell to the nearest topographic feature, such as a peak or ridge. The reasoning behind this feature is that certain tree cover types might be more prevalent near prominent topographic landmarks. For example, certain tree species might be more likely to grow on elevated areas like ridges, while others thrive in valleys or flat terrains.

##### Distance to Nearest Topographic Feature

In [ ]:
#Distance to Nearest Topographic Feature
df['D_to_N_T_F'] = np.sqrt((df['Elevation'] - df['Elevation'].min())**2)
df['D_to_N_T_F']

###### Hillshade Variability:


This feature would represent the variation in hillshade across different times of the day. It could be computed as the standard deviation of the three Hillshade indices (Hillshade_9am, Hillshade_Noon, Hillshade_3pm). The intuition is that certain tree species may prefer specific lighting conditions, and this variability could provide valuable information for classification.



In [ ]:
# with calculating the standard deviation of the three Hillshade indices

In [ ]:
hillshade_indices = ['Hillshade_9am', 'Hillshade_Noon', 'Hillshade_3pm']
df['hillshade_variability'] = df[hillshade_indices].std(axis=1)

df['hillshade_variability']

##### Topographic Wetness

 feature calculates the topographic wetness index (TWI), which represents the water availability in the area. This can capture the water stress on different tree species. The benefit of this feature is that it can provide a more direct measure of the water availability, rather than just looking at the distance to water features.


In [ ]:
#calculate topographic wetness index (TWI)

In [ ]:
m = 10 # slope gradient
A = (df['Elevation'] + df['Vertical_Distance_To_Hydrology']) / 2
S = np.arctan(df['Slope'] / 100)
TW = np.log10((A / np.sin(S)) / m)
df['TW'] = TW
df['TW'] 

After add new Features must be check null data

In [ ]:
#check is null data
df.isnull().sum()

the data type inf (infinity) when performing certain mathematical operations that result in values that exceed the representable range for floating-point numbers 


In [ ]:
np.isinf(df.values).sum()

There are 5 elements of type inf

In [ ]:
#replace it 'inf' with 0
df.replace(np.inf, 0, inplace=True)

In [ ]:
# claculate to impotance features

In [ ]:
# split  data  and target 
X = df.drop("Cover_Type", axis=1)
y = df["Cover_Type"]

In [ ]:
# creat model  LGBMClassifier
model=LGBMClassifier()
#split data 
x_train,x_test ,y_train , y_test=train_test_split(X, y ,test_size=0.30 , random_state=1)

# recursive feature elimination and fit data 
rfe = RFE(model, n_features_to_select=20)
rfe.fit(x_train , y_train )

# get the most important features
important_features = x_train.columns[rfe.support_]
    
impotance_features= important_features.to_list()
impotance_features

In [ ]:
print(impotance_features)

### Extract the correlation matrix¶


In [ ]:
plt.figure(figsize = (15, 15))
sns.heatmap(X[impotance_features].corr(), annot = True)
plt.show()

##### Plotting feature importances

In [ ]:
# Plotting feature importances
#plt.figure(figsize=(10, 6))
plt.bar(range(len(impotance_features)), rfe.ranking_[:len(impotance_features)])
plt.xlabel("Features")
plt.ylabel("Importance")
plt.xticks(range(len(impotance_features)), impotance_features, rotation=90)
plt.show()

#### relationship between elevation and cover type

In [ ]:
# Scatter plot with different colors for each cover type
sns.scatterplot(x='Elevation', y='Cover_Type', data=df, hue='Cover_Type', palette='viridis')
plt.xlabel('Elevation')
plt.ylabel('Cover Type')
plt.title('relationship between Elevation and Cover Type')
plt.show()

In [ ]:
# Box plot to visualize the distribution of elevation for each cover type
sns.boxplot(x='Cover_Type', y='Elevation', data=df, palette='viridis')
plt.xlabel('Cover Type')
plt.ylabel('Elevation')
plt.title('Distribution of Elevation for Each Cover Type')
plt.show()


In [ ]:
# Violin plot to see the density distribution of elevation for each cover type
sns.violinplot(x='Cover_Type', y='Elevation', data=df, palette='viridis')
plt.xlabel('Cover Type')
plt.ylabel('Elevation')
plt.title('Density Distribution of Elevation for Each Cover Type')
plt.show()

In [ ]:
# Heatmap to visualize the correlation between elevation and cover type
correlation_matrix = df[['Elevation', 'Cover_Type']].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation between Elevation and Cover Type')
plt.show()

#### split data 

In [ ]:
df.reset_index(drop=True,inplace=True)
X = df.drop("Cover_Type", axis=1)
y = df["Cover_Type"]

It seems like the y labels in dataset are encoded with values from 1 to 7, but XGBoost expects class labels to start from 0. To fix this issue, you need to ensure that the class labels in y start from 0 instead of 1.

In [ ]:
y = y - 1

In [ ]:
#  StandardScaler class to normalize the data with a mean of 0 and a standard deviation of 1

Stand= StandardScaler()
 
X = pd.DataFrame(Stand.fit_transform(X) , columns=X.columns)
X

In [ ]:
#spit data into  train and test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

### model XGBoost 

 find  optimal parameter  from a given set of parameters in a Grid SearchCV

#####I was going to use GridSearchCV, because it's short, but it would take a very long time in hours

###### I will find the best for the separate parametres

##### n_estimators

In [ ]:
n_estimators_range = [20,50,100,200,500]

#  cross-validation to evaluate with  n_estimators 
cv_scores = []
for n_estimators in n_estimators_range:
    model = XGBClassifier(n_estimators=n_estimators)
    scores = cross_val_score(model, X_train, y_train, cv=5)
    cv_scores.append(scores.mean())

# get  best n_estimators
best_n_estimators = n_estimators_range[np.argmax(cv_scores)]

# plot  cross-validation scores 
plt.plot(n_estimators_range, cv_scores)
plt.xlabel('n_estimators')
plt.ylabel('Cross-Validation Accuracy')
plt.title('Cross-Validation Scores : n_estimators ')
plt.show()

print("Best n_estimators:", best_n_estimators)

Best parameter to n_estimators : 500

##### max_depth

In [ ]:
max_depth_range = [None,5,10,15,20,50]

#  cross-validation to evaluate  model 
cv_scores = []
for max_depth in max_depth_range:
    model = XGBClassifier(max_depth=max_depth)
    scores = cross_val_score(model, X_train, y_train, cv=5)
    cv_scores.append(scores.mean())

# get the best max_depth 
best_max_depth = max_depth_range[np.argmax(cv_scores)]

# plot the cross-validation scores
plt.plot(max_depth_range, cv_scores)
plt.xlabel('max_depth')
plt.ylabel('Cross-Validation Accuracy')
plt.title('Cross-Validation Scores : Max_depth ')
plt.show()

print("Best max_depth:", best_max_depth)

Best parameter to max_depth : 20

##### booster

In [ ]:
booster_range = ['dart','gblinear','gbtree']

# cross-validation to evaluate the model
cv_scores = []
for booster in booster_range:
    model = XGBClassifier(booster=booster)
    scores = cross_val_score(model, X_train, y_train, cv=5)
    cv_scores.append(scores.mean())

# get the best max_depth 
best_booster = booster_range[np.argmax(cv_scores)]

# plot the cross-validation scores
plt.plot(booster_range, cv_scores)
plt.xlabel('booster')
plt.ylabel('Cross-Validation Accuracy')
plt.title('Cross-Validation Scores : Booster ')
plt.show()

print("Best booster:", best_booster)

Best parameter to Booster : dart

##### learning_rate

In [ ]:
learning_rate_range = [ 0.0001   ,0.001 ,0.01,0.1,1,10]

# cross-validation to evaluate the model
cv_scores = []
for learning_rate in learning_rate_range:
    model = XGBClassifier(learning_rate=learning_rate)
    scores = cross_val_score(model, X_train, y_train, cv=5)
    cv_scores.append(scores.mean())

# get the best max_depth 
best_learning_rate = learning_rate_range[np.argmax(cv_scores)]

# plot the cross-validation scores
plt.plot(learning_rate_range, cv_scores)
plt.xlabel('learning_rate')
plt.ylabel('Cross-Validation Accuracy')
plt.title('Cross-Validation Scores ')
plt.show()

print("Best learning_rate:", best_learning_rate)

In [ ]:
#Model XGBClassifier
model = XGBClassifier(n_estimators=best_n_estimators ,max_depth=best_max_depth , booster=best_booster ,learning_rate=best_learning_rate )

#fit data model 
model.fit(X_train, y_train)

# predictions
y_pred = model.predict(X_test)

#accuracy
accuracy = accuracy_score(y_test, y_pred)

In [ ]:
print("Accuracy model XGBClassifier  it : {:.3f}".format(accuracy))

In [ ]:
#  classification report
print(classification_report(y_test, y_pred))

In [ ]:
#confusion matrix
confusion_matrix(y_test, y_pred )

### model LightGBM 

 find  optimal parameter  from a given set of parameters in a Grid Search CV

###### I will find the best for the separate parametres

##### n_estimators

In [ ]:
n_estimators_range = [20,50,100,200,500]

#  cross-validation to evaluate with  n_estimators 
cv_scores = []
for n_estimators in n_estimators_range:
    model = LGBMClassifier(n_estimators=n_estimators)
    scores = cross_val_score(model, X_train, y_train, cv=5)
    cv_scores.append(scores.mean())

# get  best n_estimators
best_n_estimators = n_estimators_range[np.argmax(cv_scores)]

# plot  cross-validation scores 
plt.plot(n_estimators_range, cv_scores)
plt.xlabel('n_estimators')
plt.ylabel('Cross-Validation Accuracy')
plt.title('Cross-Validation Scores : n_estimators ')
plt.show()

print("Best n_estimators:", best_n_estimators)

Best parameter to n_estimators : 500

##### max_depth

In [ ]:
max_depth_range = [None,5,10,15,20,50]

#  cross-validation to evaluate  model 
cv_scores = []
for max_depth in max_depth_range:
    model = LGBMClassifier(max_depth=max_depth)
    scores = cross_val_score(model, X_train, y_train, cv=5)
    cv_scores.append(scores.mean())

# get the best max_depth 
best_max_depth = max_depth_range[np.argmax(cv_scores)]

# plot the cross-validation scores
plt.plot(max_depth_range, cv_scores)
plt.xlabel('max_depth')
plt.ylabel('Cross-Validation Accuracy')
plt.title('Cross-Validation Scores : Max_depth ')
plt.show()

print("Best max_depth:", best_max_depth)

Best parameter to max_depth : 20

##### learning_rate

In [ ]:
learning_rate_range = [ 0.0001   ,0.001 ,0.01,0.1,1,10]

# cross-validation to evaluate the model
cv_scores = []
for learning_rate in learning_rate_range:
    model = LGBMClassifier(learning_rate=learning_rate)
    scores = cross_val_score(model, X_train, y_train, cv=5)
    cv_scores.append(scores.mean())

# get the best max_depth 
best_learning_rate = learning_rate_range[np.argmax(cv_scores)]

# plot the cross-validation scores
plt.plot(learning_rate_range, cv_scores)
plt.xlabel('learning_rate')
plt.ylabel('Cross-Validation Accuracy')
plt.title('Cross-Validation Scores ')
plt.show()

print("Best learning_rate:", best_learning_rate)

In [ ]:
#Model XGBClassifier
model = LGBMClassifier(n_estimators=best_n_estimators ,max_depth=best_max_depth  ,learning_rate=best_learning_rate )

#fit data model 
model.fit(X_train, y_train)

# predictions
y_pred = model.predict(X_test)

#accuracy
accuracy = accuracy_score(y_test, y_pred)

In [ ]:
print("Accuracy model XGBClassifier  it : {:.3f}".format(accuracy))

In [ ]:
#  classification report

print(classification_report(y_test, y_pred))

In [ ]:
#confusion matrix
confusion_matrix(y_test, y_pred )

### Predict from test data

##### The same features should be added to the test data

In [ ]:
#create data frame
test=pd.DataFrame()

In [ ]:
#copy id from file test 
test['Id']= df_test['Id']
test['Id']

In [ ]:
df_test['D_to_N_T_F'] = np.sqrt((df_test['Elevation'] - df_test['Elevation'].min())**2)

In [ ]:
hillshade_indices = ['Hillshade_9am', 'Hillshade_Noon', 'Hillshade_3pm']
df_test['hillshade_variability'] = df_test[hillshade_indices].std(axis=1)

m = 10 # slope gradient
A = (df_test['Elevation'] + df_test['Vertical_Distance_To_Hydrology']) / 2
S = np.arctan(df['Slope'] / 100)
TW = np.log10((A / np.sin(S)) / m)
df_test['TW'] = TW

df_test.replace(np.inf, 0, inplace=True)


In [ ]:
#drop it columns Id
df_test.drop(['Id'],axis=1 ,  inplace=True)

In [ ]:
#Normalize data
Stand= StandardScaler()
 
df_test = pd.DataFrame(Stand.fit_transform(df_test) , columns=df_test.columns)


In [ ]:
#model XGB
model = XGBClassifier(n_estimators=best_n_estimators   )

#fit data model 
model.fit(X_train, y_train)

In [ ]:
#predicted 
test['Cover_Type']=model.predict(df_test)

In [ ]:
class_= {
    0: 1,
    1: 2,
    2: 3,
    3: 4,
    4: 5,
    5: 6,
    6: 7
}

test['Cover_Type'] = test['Cover_Type'].replace(class_)

In [ ]:
# save data frame test to file csv
test.to_csv("Sub_test.csv",sep=',' , index=False)